# AMR Soft Prompting — All-Run Results Comparison

Pulls **live** final-epoch validation metrics from Weights & Biases (`wandb`)
for every training run in the project and lays them out side by side:

- **V1** (multi-head, pre-restructure): the 2 retrained post-leakage-fix checkpoints
- **V2 core** (single-head, Runs 1–3): 6 runs — 3 tasks × {internal, external} injection
- **V2 negative controls**: 6 runs — same 3 tasks × {internal, external}, but with
  `NullSoftPrompt` (a fixed all-zero token) instead of real conditioning

That's **14 runs** total. No local checkpoints, `outputs/`, or repo clone are needed —
everything here comes from wandb run summaries/history, so this notebook runs standalone
in Colab.

This notebook also produces the **control-vs-conditioned comparison** flagged in
`docs/STATUS.md` under "What's Not Started" (item 1) — as of this writing that
comparison had numbers scattered across STATUS.md but had never been placed
side-by-side or interpreted in one place.

**Caveat carried over from STATUS.md:** every metric below is a *training-loop
final-epoch validation metric*, not a held-out `evaluate.py` pass. None of the 12 V2
runs have been run through `evaluate.py` yet.

## Setup

In [ ]:
!pip install -q wandb

import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

wandb.login()  # paste your API key from https://wandb.ai/authorize when prompted


In [ ]:
# Reference palette (matches notebooks/exploration/poster_figures.ipynb)
BLUE = "#2a78d6"
ORANGE = "#eb6834"
AQUA = "#1baf7a"
VIOLET = "#4a3aa7"
GREY = "#898781"
CRITICAL = "#d03b3b"
INK = "#0b0b0b"
INK2 = "#52514e"

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK2,
    "ytick.color": INK2,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "savefig.facecolor": "#fcfcfb",
})
FIGSIZE = (9, 5.5)

# wandb project location (see logs/*.log for the entity these runs were synced under)
ENTITY = "aidan-j-nguyen-san-jose-state-university"
PROJECT = "amr-soft-prompting"

api = wandb.Api()


def run_path(run_id: str) -> str:
    return f"{ENTITY}/{PROJECT}/{run_id}"


## Run registry

Every run this project has produced usable final numbers for, with its task, injection
mode, conditioning type, and the wandb summary key(s) that hold its headline metric.
Run IDs and metric key names are taken from `notebooks/exploration/poster_figures.ipynb`
(core V2 + V1) and from `wandb: 🚀 View run at ...` lines in `logs/*noconditioning*.log`
(negative controls).

In [ ]:
RUNS = [
    # --- V1 (multi-head, pre-restructure; only amr_gene_family is an honest metric) ---
    {"task": "amr_gene_family", "run_type": "V1", "conditioning": "mechanism+drug_class",
     "injection_mode": "internal", "run_id": "i7o4eg5n", "primary_key": "amr_gene_family_accuracy"},
    {"task": "amr_gene_family", "run_type": "V1", "conditioning": "mechanism+drug_class",
     "injection_mode": "external", "run_id": "2rr2h1f9", "primary_key": "amr_gene_family_accuracy"},

    # --- V2 Run 1: drug_class <- amr_gene_family (BCE, multi-label) ---
    {"task": "drug_class", "run_type": "core", "conditioning": "amr_gene_family",
     "injection_mode": "internal", "run_id": "mlvam3jw",
     "primary_key": "drug_class_subset_accuracy", "secondary_key": "drug_class_f1_micro"},
    {"task": "drug_class", "run_type": "core", "conditioning": "amr_gene_family",
     "injection_mode": "external", "run_id": "p9f8pdmn",
     "primary_key": "drug_class_subset_accuracy", "secondary_key": "drug_class_f1_micro"},
    {"task": "drug_class", "run_type": "control", "conditioning": "none",
     "injection_mode": "internal", "run_id": "u2vn8z4f",
     "primary_key": "drug_class_subset_accuracy", "secondary_key": "drug_class_f1_micro"},
    {"task": "drug_class", "run_type": "control", "conditioning": "none",
     "injection_mode": "external", "run_id": "v6fxnw1a",
     "primary_key": "drug_class_subset_accuracy", "secondary_key": "drug_class_f1_micro"},

    # --- V2 Run 2: resistance_mechanism <- amr_gene_family (CE) ---
    {"task": "resistance_mechanism", "run_type": "core", "conditioning": "amr_gene_family",
     "injection_mode": "internal", "run_id": "xpyy4zde", "primary_key": "resistance_mechanism_accuracy"},
    {"task": "resistance_mechanism", "run_type": "core", "conditioning": "amr_gene_family",
     "injection_mode": "external", "run_id": "oe13u15r", "primary_key": "resistance_mechanism_accuracy"},
    {"task": "resistance_mechanism", "run_type": "control", "conditioning": "none",
     "injection_mode": "internal", "run_id": "pmt33rte", "primary_key": "resistance_mechanism_accuracy"},
    {"task": "resistance_mechanism", "run_type": "control", "conditioning": "none",
     "injection_mode": "external", "run_id": "7zk50s9l", "primary_key": "resistance_mechanism_accuracy"},

    # --- V2 Run 3: amr_gene_family <- ta_proximity (CE) ---
    {"task": "amr_gene_family", "run_type": "core", "conditioning": "ta_proximity",
     "injection_mode": "internal", "run_id": "x7f7gvh3", "primary_key": "amr_gene_family_accuracy"},
    {"task": "amr_gene_family", "run_type": "core", "conditioning": "ta_proximity",
     "injection_mode": "external", "run_id": "p4kl9z5r", "primary_key": "amr_gene_family_accuracy"},
    {"task": "amr_gene_family", "run_type": "control", "conditioning": "none",
     "injection_mode": "internal", "run_id": "0u0vne25", "primary_key": "amr_gene_family_accuracy"},
    {"task": "amr_gene_family", "run_type": "control", "conditioning": "none",
     "injection_mode": "external", "run_id": "qbkr6mwu", "primary_key": "amr_gene_family_accuracy"},
]

print(f"{len(RUNS)} runs registered")


## Pull live metrics from wandb

In [ ]:
records = []
for r in RUNS:
    run = api.run(run_path(r["run_id"]))
    summary = run.summary
    primary_val = summary.get(f"val/{r['primary_key']}")
    secondary_val = summary.get(f"val/{r['secondary_key']}") if "secondary_key" in r else None
    records.append({
        "task": r["task"],
        "run_type": r["run_type"],
        "conditioning": r["conditioning"],
        "injection_mode": r["injection_mode"],
        "run_id": r["run_id"],
        "wandb_name": run.name,
        "state": run.state,
        "primary_metric": r["primary_key"],
        "primary_value": primary_val,
        "secondary_metric": r.get("secondary_key"),
        "secondary_value": secondary_val,
        "epochs_logged": summary.get("epoch"),
    })

df = pd.DataFrame(records)
df


In [ ]:
# Persist the pulled table so it survives without re-hitting the API
df.to_csv("run_comparison_results.csv", index=False)
print("Saved run_comparison_results.csv")
df.style.format({"primary_value": "{:.4f}", "secondary_value": "{:.4f}"}).background_gradient(
    subset=["primary_value"], cmap="Blues"
)


## Figure 1 — Core injection-mode ablation, by task

Internal vs. external injection for each of the 3 V2 core tasks (real conditioning,
not the negative controls).

In [ ]:
core = df[df["run_type"] == "core"]
task_order = ["drug_class", "resistance_mechanism", "amr_gene_family"]
task_labels = ["Drug Class\n(subset acc.)", "Mechanism\n(accuracy)", "Gene Family\n(<- TA-proximity)"]

internal_vals = [core[(core.task == t) & (core.injection_mode == "internal")]["primary_value"].iloc[0] for t in task_order]
external_vals = [core[(core.task == t) & (core.injection_mode == "external")]["primary_value"].iloc[0] for t in task_order]

x = np.arange(len(task_order))
width = 0.35

fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
b1 = ax.bar(x - width / 2, internal_vals, width, label="Internal", color=BLUE)
b2 = ax.bar(x + width / 2, external_vals, width, label="External", color=ORANGE)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h*100:.1f}%", (bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    fontsize=11, color=INK)

ax.set_xticks(x)
ax.set_xticklabels(task_labels, fontsize=12)
ax.set_ylabel("Validation metric", fontsize=12)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title("V2 core runs: internal vs. external injection", fontsize=14)
ax.legend(fontsize=11)
plt.show()


## Figure 2 — Conditioned vs. negative control, by task and injection mode

The actual point of the negative-control runs: does real conditioning
(`amr_gene_family` or `ta_proximity`) beat a fixed, non-learned all-zero token
(`NullSoftPrompt`)? A task/mode where the control matches or beats its conditioned
counterpart means that run's conditioning input isn't pulling its weight.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True, sharey=True)

for ax, mode in zip(axes, ["internal", "external"]):
    cond_vals = [df[(df.task == t) & (df.run_type == "core") & (df.injection_mode == mode)]["primary_value"].iloc[0] for t in task_order]
    ctrl_vals = [df[(df.task == t) & (df.run_type == "control") & (df.injection_mode == mode)]["primary_value"].iloc[0] for t in task_order]

    xm = np.arange(len(task_order))
    b1 = ax.bar(xm - width / 2, cond_vals, width, label="Conditioned", color=VIOLET)
    b2 = ax.bar(xm + width / 2, ctrl_vals, width, label="Control (zero token)", color=GREY)
    for bars in (b1, b2):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h*100:.1f}%", (bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                        fontsize=10, color=INK)

    ax.set_xticks(xm)
    ax.set_xticklabels(task_labels, fontsize=11)
    ax.set_title(f"{mode.capitalize()} injection", fontsize=13)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=10)

axes[0].set_ylabel("Validation metric", fontsize=12)
fig.suptitle("Conditioned vs. negative control, per task", fontsize=15)
plt.show()


### Delta table (conditioned − control)

In [ ]:
delta_rows = []
for t, label in zip(task_order, ["Drug Class", "Mechanism", "Gene Family (<- TA-proximity)"]):
    for mode in ["internal", "external"]:
        cond = df[(df.task == t) & (df.run_type == "core") & (df.injection_mode == mode)]["primary_value"].iloc[0]
        ctrl = df[(df.task == t) & (df.run_type == "control") & (df.injection_mode == mode)]["primary_value"].iloc[0]
        delta_rows.append({
            "task": label, "injection_mode": mode,
            "conditioned": cond, "control": ctrl, "delta": cond - ctrl,
        })

delta_df = pd.DataFrame(delta_rows)
delta_df.style.format({"conditioned": "{:.4f}", "control": "{:.4f}", "delta": "{:+.4f}"}).background_gradient(
    subset=["delta"], cmap="RdYlGn", vmin=-0.05, vmax=0.05
)


In [ ]:
print("Interpretation flags (see docs/STATUS.md \"Open Questions\"):\n")
for _, row in delta_df.iterrows():
    flag = ""
    if abs(row["delta"]) < 0.005:
        flag = "  <-- conditioning adds ~nothing over the zero token here"
    print(f"  {row.task:32s} {row.injection_mode:8s}  delta={row.delta:+.4f}{flag}")

t1 = delta_df[delta_df.task == "Drug Class"]
if len(t1) == 2 and abs(t1.iloc[0]["control"] - t1.iloc[1]["control"]) < 1e-6:
    print(
        "\nNote: Task 1 (drug_class) control accuracy is identical between internal and "
        "external injection -- flagged in STATUS.md as needing confirmation this is expected "
        "for an all-zero token (rather than injection_mode being a no-op for the null case)."
    )


## Figure 3 — Gene-family accuracy across architecture versions

`amr_gene_family` is the one target scored consistently from V1 through V2's Run 3
and its control, so it's the only metric directly comparable across the whole project
history.

In [ ]:
gf = df[df.task == "amr_gene_family"].copy()
gf["group"] = gf["run_type"].map({"V1": "V1 (multi-head)", "core": "V2 Run 3 (TA-proximity)", "control": "V2 Run 3 control (zero token)"})

groups = ["V1 (multi-head)", "V2 Run 3 (TA-proximity)", "V2 Run 3 control (zero token)"]
colors = [GREY, VIOLET, AQUA]

xg = np.arange(len(groups))
width = 0.35

internal_gf = [gf[(gf.group == g) & (gf.injection_mode == "internal")]["primary_value"].iloc[0] for g in groups]
external_gf = [gf[(gf.group == g) & (gf.injection_mode == "external")]["primary_value"].iloc[0] for g in groups]

fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
b1 = ax.bar(xg - width / 2, internal_gf, width, label="Internal", color=BLUE)
b2 = ax.bar(xg + width / 2, external_gf, width, label="External", color=ORANGE)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h*100:.1f}%", (bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    fontsize=10, color=INK)

ax.set_xticks(xg)
ax.set_xticklabels(groups, fontsize=11)
ax.set_ylabel("amr_gene_family accuracy", fontsize=12)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title("Gene-family accuracy: V1 vs. V2 Run 3 vs. its control", fontsize=14)
ax.legend(fontsize=11)
plt.show()


## Convergence curves — conditioned vs. control

Per-epoch validation curves for each task, conditioned run vs. its control, one
subplot per injection mode. A control that converges to the same accuracy on a
similar trajectory (not just the same final number) is stronger evidence that
conditioning isn't contributing signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True, sharex=True)

metric_key_by_task = {
    "drug_class": "drug_class_subset_accuracy",
    "resistance_mechanism": "resistance_mechanism_accuracy",
    "amr_gene_family": "amr_gene_family_accuracy",
}

for ax, t, label in zip(axes, task_order, task_labels):
    key = f"val/{metric_key_by_task[t]}"
    for mode, style in zip(["internal", "external"], ["-", "--"]):
        cond_id = df[(df.task == t) & (df.run_type == "core") & (df.injection_mode == mode)]["run_id"].iloc[0]
        ctrl_id = df[(df.task == t) & (df.run_type == "control") & (df.injection_mode == mode)]["run_id"].iloc[0]

        cond_hist = api.run(run_path(cond_id)).history(keys=["epoch", key], pandas=True)
        ctrl_hist = api.run(run_path(ctrl_id)).history(keys=["epoch", key], pandas=True)

        ax.plot(cond_hist["epoch"], cond_hist[key], style, color=VIOLET, linewidth=2,
                label=f"Conditioned ({mode})")
        ax.plot(ctrl_hist["epoch"], ctrl_hist[key], style, color=GREY, linewidth=2,
                label=f"Control ({mode})")

    ax.set_title(label.replace("\n", " "), fontsize=12)
    ax.set_xlabel("Epoch", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

axes[0].set_ylabel("Validation metric", fontsize=12)
axes[0].legend(fontsize=9, loc="lower right")
fig.suptitle("Convergence: conditioned vs. control", fontsize=15)
plt.show()


## Summary

- **Table** above (`df`, saved to `run_comparison_results.csv`) has every run's
  final validation metric, task, conditioning input, and injection mode.
- **Figure 1** reproduces the poster's core internal-vs-external ablation.
- **Figure 2 + delta table** are the control-vs-conditioned comparison that
  `docs/STATUS.md` flags as not yet written up — read the delta column per task
  before citing any V2 core result as evidence conditioning helps.
- **Figure 3** places V1 and V2 Run 3 (+ its control) on one gene-family-accuracy axis.
- **Convergence curves** show whether a control's final number came from a similar
  trajectory to its conditioned counterpart, or arrived differently.

**Known open flags to weigh when reading these numbers** (from `docs/STATUS.md`):
1. Run 2 (`resistance_mechanism`) sits at ~99.5% in both injection modes and both
   conditioned/control variants — matches the *shape* of the pre-fix leakage bug,
   even though the leakage mechanism itself (ground-truth-as-conditioning) was
   removed for V2. Not yet checked whether `resistance_mechanism` is just trivially
   predictable from `amr_gene_family` alone (a real correlation, not leakage, would
   also produce this).
2. Task 1 (`drug_class`) negative-control accuracy is identical between internal and
   external injection — plausible for an all-zero token, but not yet confirmed as
   expected vs. a wiring bug making `injection_mode` a no-op for the null case.
3. None of the 12 V2 runs (6 core + 6 control) have been run through the held-out
   `evaluate.py` pass yet — everything here is a training-loop final-epoch metric.